# 基于 OpenCV 的颜色追踪
在本章教程中我们会在 OpenCV 的相关功能中加入一些控制外设的函数，例如，在本章教程中，摄像头云台会转动，确保你的手或其它易碎物品远离摄像头云台的转动半径。

## 准备工作
由于产品开机默认会自动运行主程序，主程序会占用摄像头资源，这种情况下是不能使用本教程的，需要结束主程序或禁止主程序自动运行后再重新启动机器人。

这里需要注意的是，由于机器人主程序中使用了多线程且由 service 配置开机自动运行，所以常规的 sudo killall python 的方法通常是不起作用的，所以我们这里介绍禁用主程序自动运行的方法。

如果你已经禁用了机器人主程序的开机自动运行，则不需要执行下面的`结束主程序`章节。

### 结束主程序
1. 点击上方本页面选项卡旁边的 “+”号，会打开一个新的名为 Launcher 的选项卡。
2. 点击 Other 内的 Terminal，打开终端窗口。
3. 在终端窗口内输入 `bash` 后按回车。
4. 现在你可以使用 Bash Shell 来控制机器人了。
5. 输入命令： `systemctl --user stop ugv-app.service`。
6. 在终端页面，命令执行完后，继续该教程的剩余部分。

## 例程
以下代码块可以直接运行：

1. 选中下面的代码块
2. 按 Shift + Enter 运行代码块
3. 观看实时视频窗口
4. 按 `STOP` 关闭实时视频，释放摄像头资源

### 如果运行时不能看到摄像头实时画面
- 需要点击上方的 Kernel - Shut down all kernels
- 关闭本章节选项卡，再次打开
- 点击 `STOP` 释放摄像头资源后重新运行代码块
- 重启设备

### 注意事项
如果使用USB摄像头则需要取消注释`frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)`这一句。

### 运行
在本章教程中，摄像头云台会转动，确保你的手或其它易碎物品远离摄像头云台的转动半径。

我们在例程中默认检测绿色小球，确保画面背景中没有绿色物体影响颜色识别功能，你也可以通过二次开发来更改检测颜色（LAB色彩空间）。

In [ ]:
import matplotlib.pyplot as plt
import cv2
from picamera2 import Picamera2
import numpy as np
from IPython.display import display, Image
import ipywidgets as widgets
import threading
import imutils
import math
from base_ctrl import BaseController
import time
from enum import Enum
from math import isnan

ase = BaseController('/dev/ttyTHS1', 115200)

# Stop button
# ================
stopButton = widgets.ToggleButton(
    value=False,
    description='Stop',
    disabled=False,
    button_style='danger', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Description',
    icon='square' # (FontAwesome names without the `fa-` prefix)
)

class PID:
    def __init__(self, kp, ki, kd, output_limits, tolerance):
        self.kp = kp
        self.ki = ki
        self.kd = kd
        self.integral = 0.0
        self.prev_error = 0.0
        self.prev_time = None
        self.output_limits = output_limits
        self.tolerance = tolerance  

    def compute(self, setpoint, measurement):
        error = setpoint - measurement
        
        if abs(error) <= self.tolerance:
            self.integral = 0.0
            self.prev_error = 0.0
            self.prev_time = time.time()
            return 0.0

        now = time.time()
        dt = 0.0 if self.prev_time is None else now - self.prev_time

        self.integral += error * dt
        derivative = 0.0 if dt == 0 else (error - self.prev_error) / dt

        output = self.kp * error + self.ki * self.integral + self.kd * derivative
        output = max(self.output_limits[0], min(self.output_limits[1], output))

        self.prev_error = error
        self.prev_time = now
        return output

pt_x = 0.0  
pt_y = 0.0 
pt_x_pid = PID(kp=0.9, ki=0.0, kd=0.1, output_limits=(-math.pi/4, math.pi/4), tolerance=0.02)
pt_y_pid = PID(kp=0.7, ki=0.0, kd=0.05, output_limits=(-math.pi/4, math.pi/4), tolerance=0.02)

pan_angle = 0
tilt_angle = 0

# Display function
# ================
def view(button):
    global pt_x, pt_y
    # picam2 = Picamera2()
    # picam2.configure(picam2.create_video_configuration(main={"format": 'XRGB8888', "size": (640, 480)}))
    # picam2.start()

    camera = cv2.VideoCapture(0) # 创建摄像头实例
    #设置分辨率
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    display_handle=display(None, display_id=True)

    color_lower = np.array([0, 0, 0])
    color_upper = np.array([255, 110, 195])
    
    while True:
        # frame = picam2.capture_array()
        # frame = cv2.flip(frame, 1) # if your camera reverses your image

        _, frame = camera.read()

        # uncomment this line if you are using USB camera
        # frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        cx, cy, w = None, None, None
        img_h, img_w = frame.shape[:2]
        center_x, center_y = img_w // 2, img_h // 2
        lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        mask = cv2.inRange(lab, color_lower, color_upper)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours:
            c = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(c)
            if area > 100:  
                ((x, y), radius) = cv2.minEnclosingCircle(c)
                circularity = area / (math.pi * radius * radius)
                if 0.5 < circularity < 1.3:  
                    cx, cy, w = int(x), int(y), int(radius*2)

                    cv2.circle(frame, (cx, cy), int(radius), (0, 255, 0), 2)
                    cv2.circle(frame, (cx, cy), 3, (255, 0, 0), -1)

        if cx is not None:
            error_x = 0.1*(cx - center_x) / center_x
            error_y = 0.1*(cy - center_y) / center_y

            delta_x = pt_x_pid.compute(0.0, error_x)
            delta_y = pt_y_pid.compute(0.0, error_y)

            pt_x += delta_x
            pt_y += delta_y

            pt_x = max(-3.14, min(3.14, pt_x))
            pt_y = max(-0.523, min(1.57, pt_y))

            pan_angle = -(180 * pt_x) / 3.14
            tilt_angle = (180 * pt_y) / 3.14

            base.base_json_ctrl({"T": 133 ,"X": pan_angle,"Y": tilt_angle,"SPD": 0,"ACC":128})
            cv2.putText(frame, f'X: {pan_angle:.2f}  Y: {tilt_angle:.2f}',(center_x+50, center_y+80), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
            
        _, frame = cv2.imencode('.jpeg', frame)
        display_handle.update(Image(data=frame.tobytes()))
        if stopButton.value==True:
            # picam2.close()
            camera.release()
            display_handle.update(None)
            
            
# Run
# ================
display(stopButton)
thread = threading.Thread(target=view, args=(stopButton,))
thread.start()